# Library DBMS — Step 5: Populate Tables
Generates and inserts ≥10 realistic tuples per table using the Faker library.
Run **after** `library_schema.ipynb` has been executed (so `library.db` and all tables already exist).

In [1]:
import sqlite3, random
from datetime import date, timedelta
from faker import Faker

fake = Faker()
Faker.seed(42)
random.seed(42)

DB_PATH = 'library.db'
con = sqlite3.connect(DB_PATH)
con.execute('PRAGMA foreign_keys = ON;')
cur = con.cursor()

# ── helpers ────────────────────────────────────────────────────────────────
def rdate(start='2015-01-01', end='2026-08-10'):
    """Return a random ISO date string between start and end."""
    s = date.fromisoformat(start)
    e = date.fromisoformat(end)
    return (s + timedelta(days=random.randint(0, (e - s).days))).isoformat()

def pick(seq, n=1):
    """Random sample without replacement; returns list."""
    return random.sample(list(seq), k=min(n, len(seq)))

def lastids(table, id_col):
    """Fetch all primary-key values from a table."""
    return [r[0] for r in cur.execute(f'SELECT {id_col} FROM {table}').fetchall()]

print('Connected. Faker ready.')

Connected. Faker ready.


## 1 — Lookup tables
Already seeded in Step 4 (`library_schema.ipynb`). Verify counts here.

In [2]:
lookups = [
    ('Type',               'type_id'),
    ('Acquisition_Status', 'acquisition_status_id'),
    ('Copy_Status',        'copy_status_id'),
    ('Member_Status',      'member_status_id'),
    ('Request_Status',     'request_status_id'),
    ('Audience',           'audience_id'),
]
for tbl, col in lookups:
    n = cur.execute(f'SELECT COUNT(*) FROM {tbl}').fetchone()[0]
    print(f'{tbl:25s}: {n} rows')

Type                     : 5 rows
Acquisition_Status       : 3 rows
Copy_Status              : 5 rows
Member_Status            : 3 rows
Request_Status           : 4 rows
Audience                 : 5 rows


## 2 — Library

In [3]:
LIBRARY_ID      = 1
LIBRARY_NAME    = 'SFU Library'
LIBRARY_ADDRESS = '8888 University Dr W, Burnaby, BC V5A 1S6'

cur.execute(
    'INSERT INTO Library (library_id, name, address) VALUES (?,?,?)',
    (LIBRARY_ID, LIBRARY_NAME, LIBRARY_ADDRESS)
)
con.commit()
library_ids = lastids('Library', 'library_id')
print(f'Library: {len(library_ids)} row inserted')

Library: 1 row inserted


## 3 — Personnel

In [4]:
ROLES = ['Librarian', 'Library Technician', 'Clerk', 'Branch Manager', 'IT Support']

personnel_rows = []
used_emails = set()
while len(personnel_rows) < 15:
    email = fake.unique.email()
    if email in used_emails:
        continue
    used_emails.add(email)
    personnel_rows.append((
        fake.first_name(),
        fake.last_name(),
        email,
        fake.phone_number()[:20],
        random.choice(ROLES),
        rdate('2010-01-01', '2023-06-01'),
        round(random.uniform(38000, 85000), 2),
        random.choice(library_ids)
    ))

cur.executemany(
    'INSERT INTO Personnel (first_name,last_name,email,phone,role,start_date,salary,library_id) VALUES (?,?,?,?,?,?,?,?)',
    personnel_rows
)
con.commit()
personnel_ids = lastids('Personnel', 'personnel_id')
print(f'Personnel: {len(personnel_ids)} rows inserted')

Personnel: 15 rows inserted


## 4 — Author

In [5]:
author_rows = [(fake.first_name(), fake.last_name()) for _ in range(15)]
cur.executemany('INSERT INTO Author (first_name, last_name) VALUES (?,?)', author_rows)
con.commit()
author_ids = lastids('Author', 'author_id')
print(f'Author: {len(author_ids)} rows inserted')

Author: 15 rows inserted


## 5 — Item

In [6]:
type_ids   = lastids('Type',               'type_id')
acq_ids    = lastids('Acquisition_Status', 'acquisition_status_id')

GENRES = ['Fiction', 'Non-Fiction', 'Mystery', 'Science Fiction', 'Biography',
          'History', 'Children', 'Self-Help', 'Documentary', 'Horror']

item_rows = []
for _ in range(15):
    item_rows.append((
        fake.catch_phrase(),          # title — quirky but readable
        random.randint(1980, 2024),
        random.choice(GENRES),
        random.choice(library_ids),
        random.choice(type_ids),
        random.choice(acq_ids)
    ))

cur.executemany(
    'INSERT INTO Item (title,publication_year,genre,library_id,type_id,acquisition_status_id) VALUES (?,?,?,?,?,?)',
    item_rows
)
con.commit()
item_ids = lastids('Item', 'item_id')
print(f'Item: {len(item_ids)} rows inserted')

Item: 15 rows inserted


## 6 — Written_By  (Item ↔ Author, M:N)

In [7]:
written_by_rows = set()
# Ensure every item has at least one author
for iid in item_ids:
    written_by_rows.add((iid, random.choice(author_ids)))
# Add ~10 extra pairs (some items co-authored)
while len(written_by_rows) < len(item_ids) + 10:
    written_by_rows.add((random.choice(item_ids), random.choice(author_ids)))

cur.executemany('INSERT OR IGNORE INTO Written_By (item_id, author_id) VALUES (?,?)', list(written_by_rows))
con.commit()
print(f'Written_By: {len(written_by_rows)} rows inserted')

Written_By: 25 rows inserted


## 7 — Item_Copy

In [8]:
copy_status_ids = lastids('Copy_Status', 'copy_status_id')
# Get the 'Available' status id — needed so trigger T3 allows loans later
available_status_id = cur.execute(
    "SELECT copy_status_id FROM Copy_Status WHERE status_name='Available'"
).fetchone()[0]

copy_rows = []
for _ in range(20):   # more copies than items — realistic
    copy_rows.append((
        rdate('2010-01-01', '2024-01-01'),
        random.choice(item_ids),
        available_status_id     # start all copies as Available; triggers will update
    ))

cur.executemany(
    'INSERT INTO Item_Copy (date_obtained, item_id, copy_status_id) VALUES (?,?,?)',
    copy_rows
)
con.commit()
copy_ids = lastids('Item_Copy', 'item_copy_id')
print(f'Item_Copy: {len(copy_ids)} rows inserted')

Item_Copy: 20 rows inserted


## 8 — Member

In [9]:
member_status_ids = lastids('Member_Status', 'member_status_id')
active_status_id  = cur.execute(
    "SELECT member_status_id FROM Member_Status WHERE status_name='Active'"
).fetchone()[0]

member_rows = []
used_member_emails = set()
while len(member_rows) < 15:
    email = fake.unique.email()
    if email in used_member_emails:
        continue
    used_member_emails.add(email)
    # Weight heavily toward Active so loans / registrations don't get blocked by T4
    status = active_status_id if random.random() < 0.8 else random.choice(member_status_ids)
    member_rows.append((
        fake.first_name(),
        fake.last_name(),
        email,
        random.choice(library_ids),
        status
    ))

cur.executemany(
    'INSERT INTO Member (first_name,last_name,email,library_id,member_status_id) VALUES (?,?,?,?,?)',
    member_rows
)
con.commit()
member_ids = lastids('Member', 'member_id')
# Keep only Active members for loan / registration inserts
active_member_ids = [
    r[0] for r in cur.execute(
        'SELECT member_id FROM Member WHERE member_status_id = ?', (active_status_id,)
    ).fetchall()
]
print(f'Member: {len(member_ids)} rows inserted ({len(active_member_ids)} active)')

Member: 15 rows inserted (15 active)


## 9 — Volunteer  (ISA subset of Member)

In [10]:
AVAILABILITY = ['Weekday mornings', 'Weekday afternoons', 'Weekends', 'Flexible']

volunteer_member_ids = pick(active_member_ids, 10)
volunteer_rows = [
    (mid, rdate('2018-01-01', '2024-06-01'), random.choice(AVAILABILITY))
    for mid in volunteer_member_ids
]
cur.executemany(
    'INSERT INTO Volunteer (member_id, start_date, availability) VALUES (?,?,?)',
    volunteer_rows
)
con.commit()
print(f'Volunteer: {len(volunteer_rows)} rows inserted')

Volunteer: 10 rows inserted


## 10 — Room

In [11]:
ROOM_NAMES = [
    'Reading Room A', 'Reading Room B', 'Community Hall', 'Study Room 1',
    'Study Room 2',   'Story Time Room', 'Conference Room', 'Media Lab',
    'Quiet Lounge',   'Maker Space'
]

room_rows = [
    (name, random.randint(10, 120), random.choice(library_ids))
    for name in ROOM_NAMES
]
cur.executemany(
    'INSERT INTO Room (room_name, capacity, library_id) VALUES (?,?,?)',
    room_rows
)
con.commit()
room_ids = lastids('Room', 'room_id')
print(f'Room: {len(room_ids)} rows inserted')

Room: 10 rows inserted


## 11 — Event
One event per (room, date) pair — satisfies trigger T6.

In [12]:
EVENT_TYPES = ['Book Club', 'Author Talk', 'Movie Screening', 'Workshop',
               'Story Time', 'Tech Help', 'Art Class', 'Language Exchange']

event_rows = []
used_room_dates = set()
attempts = 0
while len(event_rows) < 12 and attempts < 500:
    attempts += 1
    room_id = random.choice(room_ids)
    ev_date = rdate('2024-01-01', '2025-06-30')
    if (room_id, ev_date) in used_room_dates:
        continue
    used_room_dates.add((room_id, ev_date))
    event_rows.append((
        fake.bs().title(),
        ev_date,
        random.choice(EVENT_TYPES),
        room_id
    ))

cur.executemany(
    'INSERT INTO Event (title, date, event_type, room_id) VALUES (?,?,?,?)',
    event_rows
)
con.commit()
event_ids = lastids('Event', 'event_id')
print(f'Event: {len(event_ids)} rows inserted')

Event: 12 rows inserted


## 12 — Targets  (Event ↔ Audience, M:N)

In [13]:
audience_ids = lastids('Audience', 'audience_id')

targets_rows = set()
for eid in event_ids:
    targets_rows.add((eid, random.choice(audience_ids)))   # at least one audience per event
while len(targets_rows) < len(event_ids) + 8:
    targets_rows.add((random.choice(event_ids), random.choice(audience_ids)))

cur.executemany('INSERT OR IGNORE INTO Targets (event_id, audience_id) VALUES (?,?)', list(targets_rows))
con.commit()
print(f'Targets: {len(targets_rows)} rows inserted')

Targets: 20 rows inserted


## 13 — Registers_For  (Member ↔ Event, M:N)
Stays under room capacity — trigger T7 enforced.

In [14]:
registers_rows = set()
for eid in event_ids:
    cap = cur.execute(
        'SELECT r.capacity FROM Event e JOIN Room r ON r.room_id=e.room_id WHERE e.event_id=?', (eid,)
    ).fetchone()[0]
    slots = min(cap, random.randint(3, 8))     # register 3-8 members or up to capacity
    for mid in pick(active_member_ids, slots):
        registers_rows.add((mid, eid, rdate('2023-12-01', '2024-12-31')))

cur.executemany(
    'INSERT OR IGNORE INTO Registers_For (member_id, event_id, registration_date) VALUES (?,?,?)',
    list(registers_rows)
)
con.commit()
print(f'Registers_For: {len(registers_rows)} rows inserted')

Registers_For: 60 rows inserted


## 14 — Loan
Only Active members, only Available copies. Triggers T3 & T4 fire automatically.

In [15]:
loan_rows = []
used_copies = set()   # can't loan the same copy twice without returning it first

available_copies = [
    r[0] for r in cur.execute(
        "SELECT item_copy_id FROM Item_Copy WHERE copy_status_id=?", (available_status_id,)
    ).fetchall()
]

for copy_id in pick(available_copies, 12):
    loan_date = date.fromisoformat(rdate('2023-06-01', '2024-09-01'))
    due_date  = loan_date + timedelta(days=random.choice([14, 21, 28]))
    # ~70% of loans are returned
    if random.random() < 0.7:
        returned_date = (loan_date + timedelta(days=random.randint(1, (due_date - loan_date).days + 10))).isoformat()
    else:
        returned_date = None
    loan_rows.append((
        loan_date.isoformat(),
        due_date.isoformat(),
        returned_date,
        random.choice(active_member_ids),
        copy_id
    ))

# Insert one at a time so trigger T3 can update copy status between inserts
for row in loan_rows:
    cur.execute(
        'INSERT INTO Loan (loan_date,due_date,returned_date,member_id,item_copy_id) VALUES (?,?,?,?,?)',
        row
    )
con.commit()
loan_ids = lastids('Loan', 'loan_id')
print(f'Loan: {len(loan_ids)} rows inserted')

Loan: 12 rows inserted


## 15 — Fine
Generated for loans that were returned late or are still overdue.

In [16]:
FINE_TYPES = ['Overdue', 'Damaged', 'Lost']

fine_rows = []
fined_loan_fine_pairs = set()

overdue_loans = cur.execute(
    '''SELECT loan_id, loan_date, due_date, returned_date FROM Loan
       WHERE returned_date > due_date OR (returned_date IS NULL AND due_date < date('now'))'''
).fetchall()

# Every overdue loan gets an Overdue fine
for loan_id, loan_date, due_date, returned_date in overdue_loans:
    pair = (loan_id, 'Overdue')
    if pair not in fined_loan_fine_pairs:
        days_late = random.randint(1, 30)
        fine_rows.append((loan_id, 'Overdue', round(days_late * 0.25, 2)))
        fined_loan_fine_pairs.add(pair)

# A few loans also get Damaged or Lost fines
extra_fine_loans = pick(loan_ids, min(5, len(loan_ids)))
for loan_id in extra_fine_loans:
    ft = random.choice(['Damaged', 'Lost'])
    pair = (loan_id, ft)
    if pair not in fined_loan_fine_pairs:
        fine_rows.append((loan_id, ft, round(random.uniform(5.0, 50.0), 2)))
        fined_loan_fine_pairs.add(pair)

# Pad to at least 10 rows
while len(fine_rows) < 10:
    loan_id = random.choice(loan_ids)
    ft = random.choice(FINE_TYPES)
    pair = (loan_id, ft)
    if pair not in fined_loan_fine_pairs:
        fine_rows.append((loan_id, ft, round(random.uniform(1.0, 40.0), 2)))
        fined_loan_fine_pairs.add(pair)

cur.executemany('INSERT OR IGNORE INTO Fine (loan_id, fine_type, amount) VALUES (?,?,?)', fine_rows)
con.commit()
print(f'Fine: {len(fine_rows)} rows inserted')

Fine: 10 rows inserted

## 16 — Payment
Partial or full payments toward fines. Trigger T5 prevents overpayment.

In [17]:
fines_in_db = cur.execute('SELECT loan_id, fine_type, amount FROM Fine').fetchall()

payment_rows = []
for loan_id, fine_type, fine_amount in random.sample(fines_in_db, min(10, len(fines_in_db))):
    # Pay between 50% and 100% of the fine
    paid = round(random.uniform(fine_amount * 0.5, fine_amount), 2)
    payment_rows.append((
        rdate('2024-01-01', '2024-12-31'),
        paid,
        loan_id,
        fine_type
    ))

cur.executemany(
    'INSERT INTO Payment (date, amount, loan_id, fine_type) VALUES (?,?,?,?)',
    payment_rows
)
con.commit()
print(f'Payment: {len(payment_rows)} rows inserted')

Payment: 10 rows inserted


## 17 — Donates
Some item copies were donated by members. Uses copies NOT already loaned.

In [18]:
loaned_copy_ids = set(r[0] for r in cur.execute('SELECT item_copy_id FROM Loan').fetchall())
donation_candidates = [c for c in copy_ids if c not in loaned_copy_ids]

donate_copies = pick(donation_candidates, min(10, len(donation_candidates)))
donate_rows = [
    (cid, random.choice(member_ids), rdate('2019-01-01', '2024-06-01'))
    for cid in donate_copies
]

cur.executemany(
    'INSERT OR IGNORE INTO Donates (item_copy_id, member_id, donation_date) VALUES (?,?,?)',
    donate_rows
)
con.commit()
print(f'Donates: {len(donate_rows)} rows inserted')

Donates: 8 rows inserted


## 18 — Help_Request

In [19]:
request_status_ids = lastids('Request_Status', 'request_status_id')
open_status_id = cur.execute(
    "SELECT request_status_id FROM Request_Status WHERE status_name='Open'"
).fetchone()[0]

REQUEST_DESCRIPTIONS = [
    'Cannot locate a book I reserved.',
    'Need help using the library catalogue system.',
    'Requesting an interlibrary loan.',
    'Issue with my library card — shows as expired.',
    'Looking for research materials on local history.',
    'DVD is scratched and unplayable.',
    'Printer in the library is not working.',
    'Request to extend loan period.',
    'Cannot access e-book with my account.',
    'Dispute a fine charged to my account.',
    'Need assistance with audio equipment in media lab.',
    'Book returned but still showing as checked out.',
]

help_rows = []
for desc in REQUEST_DESCRIPTIONS:
    # ~60% of requests are assigned to a staff member
    assigned = random.choice(personnel_ids) if random.random() < 0.6 else None
    status   = open_status_id if assigned is None else random.choice(request_status_ids)
    help_rows.append((
        rdate('2023-01-01', '2024-12-31'),
        desc,
        random.choice(member_ids),
        assigned,
        status
    ))

cur.executemany(
    'INSERT INTO Help_Request (request_date,description,member_id,personnel_id,request_status_id) VALUES (?,?,?,?,?)',
    help_rows
)
con.commit()
print(f'Help_Request: {len(help_rows)} rows inserted')

Help_Request: 12 rows inserted


## 19 — Verification: row counts for all 23 tables

In [20]:
ALL_TABLES = [
    'Library', 'Personnel', 'Type', 'Acquisition_Status', 'Item', 'Author',
    'Written_By', 'Copy_Status', 'Item_Copy', 'Member_Status', 'Member',
    'Volunteer', 'Loan', 'Fine', 'Payment', 'Donates', 'Room', 'Event',
    'Audience', 'Targets', 'Registers_For', 'Request_Status', 'Help_Request'
]

print(f'{"Table":<25} {"Rows":>6}  {"≥10?":>5}')
print('-' * 42)
all_ok = True
for tbl in ALL_TABLES:
    n = cur.execute(f'SELECT COUNT(*) FROM {tbl}').fetchone()[0]
    ok = '✓' if n >= 10 else '✗'
    if n < 10:
        all_ok = False
    print(f'{tbl:<25} {n:>6}  {ok:>5}')

print()
print('All tables have ≥10 rows:', '✓ YES' if all_ok else '✗ NO — check flagged tables')

Table                       Rows   ≥10?
------------------------------------------
Library                        1      ✗
Personnel                     15      ✓
Type                           5      ✗
Acquisition_Status             3      ✗
Item                          15      ✓
Author                        15      ✓
Written_By                    25      ✓
Copy_Status                    5      ✗
Item_Copy                     20      ✓
Member_Status                  3      ✗
Member                        15      ✓
Volunteer                     10      ✓
Loan                          12      ✓
Fine                          10      ✓
Payment                       10      ✓
Donates                        8      ✗
Room                          10      ✓
Event                         12      ✓
Audience                       5      ✗
Targets                       20      ✓
Registers_For                 60      ✓
Request_Status                 4      ✗
Help_Request                  12     

In [21]:
# FK integrity check
fk_violations = cur.execute('PRAGMA foreign_key_check;').fetchall()
if fk_violations:
    print('FK violations:', fk_violations)
else:
    print('No FK violations — referential integrity confirmed.')

con.close()
print('Done.')

No FK violations — referential integrity confirmed.
Done.
